Bulk delete all the subscription emails after processing them, we need to add functionality to move the emails to the trash folder. Here is the updated script that includes both logging and bulk deletion:

Updated Python Script with Logging and Bulk Deletion

Steps to Generate an App-Specific Password:

Generate an App-Specific Password:

    Go to your Google Account: https://myaccount.google.com/security?hl=en_GB&utm_source=OGB&utm_medium=act
    Select "Security" on the left navigation panel.
    Under "Signing in to Google," select "App passwords."
    You might need to sign in again.
    Select the app and device you want to generate the app password for.
    Follow the on-screen instructions to generate the password. This will be a 16-character password.
    https://myaccount.google.com/apppasswords?pli=1&rapt=AEjHL4Pw3n7_18Ff0UO6M-jdEu_cSHHY02uZJw0jbk_RAOjZp1NCicBVUAi87TMPVCHfMV3VYzzQu89ekwlYLz8crjAUk844xnuLWlRBv3aIMSccSpaEW68

In [18]:
import imaplib
import logging

# Configure logging
logging.basicConfig(filename='unsubscribe_log.txt', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Account credentials
username = "ilo.laz12@gmail.com"
password = "hjwh lblz fnmq fjyl"  # Use the generated app-specific password

# Step 1: Connect to the mail server
try:
    mail = imaplib.IMAP4_SSL("imap.gmail.com")
    mail.login(username, password)
    print("Connected to the mail server.")
    logging.info("Connected to the mail server.")
except Exception as e:
    print(f"Failed to connect to the mail server: {e}")
    logging.error(f"Failed to connect to the mail server: {e}")

Connected to the mail server.


In [19]:
# Step 2: Select the inbox
try:
    mail.select("inbox")
    print("Inbox selected.")
    logging.info("Inbox selected.")
except Exception as e:
    print(f"Failed to select the inbox: {e}")
    logging.error(f"Failed to select the inbox: {e}")


Inbox selected.


In [20]:
# Step 3: Fetch email IDs
try:
    status, messages = mail.search(None, "ALL")
    if status != "OK":
        print("Failed to retrieve emails.")
        logging.error("Failed to retrieve emails.")
    else:
        email_ids = messages[0].split()
        print(f"Found {len(email_ids)} emails.")
        logging.info(f"Found {len(email_ids)} emails.")
except Exception as e:
    print(f"Failed to fetch email IDs: {e}")
    logging.error(f"Failed to fetch email IDs: {e}")


Found 17717 emails.


In [28]:
# Step 4: Fetch and parse multiple emails
try:
    num_emails_to_check = 10  # Limit the number of emails for testing
    for email_id in email_ids[:num_emails_to_check]:
        status, msg_data = mail.fetch(email_id, "(RFC822)")
        if status != "OK":
            print(f"Failed to fetch email ID {email_id}.")
            logging.warning(f"Failed to fetch email ID {email_id}.")
        else:
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        subject = subject.decode(encoding if encoding else "utf-8")
                    print(f"Email subject: {subject}")
                    logging.info(f"Email subject: {subject}")
except Exception as e:
    print(f"Failed to fetch and parse emails: {e}")
    logging.error(f"Failed to fetch and parse emails: {e}")

Email subject: Lesson Confirmation
Email subject: Your Application
Email subject: Account details for ilo_laz at www.jobswithus.com.au
Email subject: Your Onetest Account Details
Email subject: RE: SEEK: Application For Project Administrator – Fortitude Valley   , Ref: , Application Source: SEEK
Email subject: Password Updated
Email subject: Your application for Vacation Student (Mechanical) is not yet
 complete
Email subject: WorkPac Contact - Megan Shorter
Email subject: avast! Account - registration confirmation request
Email subject: Thanks for applying to Google


In [29]:
# Step 5: Unsubscribe and move emails to trash
try:
    for email_id in email_ids[:num_emails_to_check]:
        status, msg_data = mail.fetch(email_id, "(RFC822)")
        if status != "OK":
            print(f"Failed to fetch email ID {email_id}.")
            logging.warning(f"Failed to fetch email ID {email_id}.")
        else:
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        subject = subject.decode(encoding if encoding else "utf-8")
                    print(f"Email subject: {subject}")
                    logging.info(f"Email subject: {subject}")
                    for part in msg.walk():
                        if part.get_content_type() == "text/html":
                            body = part.get_payload(decode=True).decode()
                            if "unsubscribe" in body.lower():
                                start = body.lower().find("unsubscribe")
                                end = body.find('"', start)
                                unsubscribe_url = body[start:end]
                                webbrowser.open(unsubscribe_url)
                                print(f"Unsubscribe link opened: {unsubscribe_url}")
                                logging.info(f"Unsubscribe link opened: {unsubscribe_url}")
                    # Move the email to the trash folder
                    mail.store(email_id, '+X-GM-LABELS', '\\Trash')
                    print(f"Email moved to trash: {subject}")
                    logging.info(f"Email moved to trash: {subject}")
except Exception as e:
    print(f"Failed to unsubscribe and move emails to trash: {e}")
    logging.error(f"Failed to unsubscribe and move emails to trash: {e}")
    
'''# Final Step: Logout from the email server
try:
    mail.logout()
    print("Logged out from the email server.")
    logging.info("Logged out from the email server.")
except Exception as e:
    print(f"Failed to log out: {e}")
    logging.error(f"Failed to log out: {e}")'''

Email subject: Lesson Confirmation
Email moved to trash: Lesson Confirmation
Email subject: Your Application
Failed to unsubscribe and move emails to trash: 'utf-8' codec can't decode byte 0x92 in position 1821: invalid start byte


'# Final Step: Logout from the email server\ntry:\n    mail.logout()\n    print("Logged out from the email server.")\n    logging.info("Logged out from the email server.")\nexcept Exception as e:\n    print(f"Failed to log out: {e}")\n    logging.error(f"Failed to log out: {e}")'

In [1]:
import imaplib
import email
from email.header import decode_header
import webbrowser
import logging

# Configure logging
logging.basicConfig(filename='unsubscribe_log.txt', level=logging.INFO,
                    format='%(asctime)s - %(levelname)s - %(message)s')

# Account credentials
username = "ilo.laz12@gmail.com"
password = "hjwh lblz fnmq fjyl"  # Use the generated app-specific password

# Step 1: Connect to the mail server
try:
    mail = imaplib.IMAP4_SSL("imap.gmail.com")
    mail.login(username, password)
    print("Connected to the mail server.")
    logging.info("Connected to the mail server.")
except Exception as e:
    print(f"Failed to connect to the mail server: {e}")
    logging.error(f"Failed to connect to the mail server: {e}")

# Step 2: Select the inbox
try:
    mail.select("inbox")
    print("Inbox selected.")
    logging.info("Inbox selected.")
except Exception as e:
    print(f"Failed to select the inbox: {e}")
    logging.error(f"Failed to select the inbox: {e}")

# Step 3: Fetch email IDs
try:
    status, messages = mail.search(None, "ALL")
    if status != "OK":
        print("Failed to retrieve emails.")
        logging.error("Failed to retrieve emails.")
    else:
        email_ids = messages[0].split()
        print(f"Found {len(email_ids)} emails.")
        logging.info(f"Found {len(email_ids)} emails.")
except Exception as e:
    print(f"Failed to fetch email IDs: {e}")
    logging.error(f"Failed to fetch email IDs: {e}")

# Helper function to decode email subject
def decode_str(s):
    for encoding in ('utf-8', 'latin1', 'ascii'):
        try:
            return s.decode(encoding)
        except (UnicodeDecodeError, AttributeError):
            continue
    return s.decode('utf-8', errors='replace')

# Step 4: Fetch and parse multiple emails
try:
    num_emails_to_check = 10  # Limit the number of emails for testing
    for email_id in email_ids[:num_emails_to_check]:
        status, msg_data = mail.fetch(email_id, "(RFC822)")
        if status != "OK":
            print(f"Failed to fetch email ID {email_id}.")
            logging.warning(f"Failed to fetch email ID {email_id}.")
        else:
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        subject = decode_str(subject)
                    print(f"Email subject: {subject}")
                    logging.info(f"Email subject: {subject}")
except Exception as e:
    print(f"Failed to fetch and parse emails: {e}")
    logging.error(f"Failed to fetch and parse emails: {e}")

# Step 5: Unsubscribe and move emails to trash
try:
    for email_id in email_ids[:num_emails_to_check]:
        status, msg_data = mail.fetch(email_id, "(RFC822)")
        if status != "OK":
            print(f"Failed to fetch email ID {email_id}.")
            logging.warning(f"Failed to fetch email ID {email_id}.")
        else:
            for response_part in msg_data:
                if isinstance(response_part, tuple):
                    msg = email.message_from_bytes(response_part[1])
                    subject, encoding = decode_header(msg["Subject"])[0]
                    if isinstance(subject, bytes):
                        subject = decode_str(subject)
                    print(f"Email subject: {subject}")
                    logging.info(f"Email subject: {subject}")
                    for part in msg.walk():
                        if part.get_content_type() == "text/html":
                            try:
                                body = part.get_payload(decode=True).decode('utf-8')
                            except UnicodeDecodeError:
                                body = part.get_payload(decode=True).decode('latin1')
                            if "unsubscribe" in body.lower():
                                start = body.lower().find("unsubscribe")
                                end = body.find('"', start)
                                unsubscribe_url = body[start:end]
                                webbrowser.open(unsubscribe_url)
                                print(f"Unsubscribe link opened: {unsubscribe_url}")
                                logging.info(f"Unsubscribe link opened: {unsubscribe_url}")
                    # Move the email to the trash folder
                    mail.store(email_id, '+X-GM-LABELS', '\\Trash')
                    print(f"Email moved to trash: {subject}")
                    logging.info(f"Email moved to trash: {subject}")
except Exception as e:
    print(f"Failed to unsubscribe and move emails to trash: {e}")
    logging.error(f"Failed to unsubscribe and move emails to trash: {e}")

# Final Step: Logout from the email server
try:
    mail.logout()
    print("Logged out from the email server.")
    logging.info("Logged out from the email server.")
except Exception as e:
    print(f"Failed to log out: {e}")
    logging.error(f"Failed to log out: {e}")


Connected to the mail server.
Inbox selected.
Found 14834 emails.
Failed to fetch and parse emails: expected string or bytes-like object, got 'NoneType'
Failed to unsubscribe and move emails to trash: expected string or bytes-like object, got 'NoneType'
Logged out from the email server.
